In [2]:
%load_ext autoreload
%autoreload 2

In [112]:
import sys

In [113]:
sys.argv = ['-f'] + ["MIC", "-a", "cpu"]

In [114]:
import os
sys.path.append("/home/pwiesenbach/BertGCN")
os.chdir("/home/pwiesenbach/BertGCN")

In [115]:
from entry import * 

In [116]:
import importlib
import entry
import logging
importlib.reload(logging)
importlib.reload(entry)

<module 'entry' from '/beegfs/homes/pwiesenbach/BertGCN/entry.py'>

In [117]:
from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer
from operator import itemgetter

In [118]:
dataset_file = Path("data") / f"medindcls_{args.bertmodel}_{args.doclevel}.json"
if not dataset_file.exists():
    print("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, task="MIC", doclevel=args.doclevel, clean=False)
    with open(dataset_file, "wb") as f:
        print(f"Saving dataset under {dataset_file}")
        pickle.dump(dataset, f)
else:
    print(f"Loading dataset from: {dataset_file}")
    with open(dataset_file, "rb") as f:
        dataset = pickle.load(f)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)
    
def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

Loading dataset from: data/medindcls_medbert_letter.json


/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator OneHotEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [119]:
MODELNAME = Path(PRETRAINEDMODEL).stem
if args.data == "MIC":
    DATASET = "med_indication_all_RF_diag"
    if args.testunklar:
        DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}_testunklar"
    else:
        DATASETPATH =  Path("data") / f"ind.{DATASET}_{args.doclevel}"
    BERTSAVEDIR = Path(f"models/finetuned/{args.doclevel}")
    if args.testunklar:
        BERTPATH = Path(f"{BERTSAVEDIR}/{MODELNAME}_med_indication_all_RF_diag_testunklar_best.pt")
    else:
        BERTPATH = Path(f"{BERTSAVEDIR}/{MODELNAME}_med_indication_all_RF_diag_best.pt")
elif args.data == "CSC":
    DATASET = "CARDIODE400_main"
    DATASETPATH =  Path("data") / f"ind.{DATASET}"
    BERTPATH = Path("models/finetuned/gbert-base_CARDIODE400_main_best.pt")

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)
doc_mask = train_mask + val_mask + test_mask
adj.sum()

16104981.51778602

In [128]:
ig_gcn_bert_path = f"models/gcn/{args.mixfactor}/{args.doclevel}/ig_attrs_gcn_med_bert_local_MIC.npz"
shap_gcn_bert_path = f"models/gcn/{args.mixfactor}/{args.doclevel}/shap_values_gcn_med_bert_local_MIC.npz"

ig_gcn_bert_values = np.load(ig_gcn_bert_path, "rb")["arr_0"]
shap_gcn_bert_values = np.load(shap_gcn_bert_path, "rb")["arr_0"]

In [129]:
ig_gcn_bert_values[:10]

array([[ 6.17208251e-03, -3.84835275e-03, -3.49862238e-03, ...,
        -3.96471453e-04, -2.54856159e-03, -2.26628204e-03],
       [-5.39291191e-03, -1.22325251e-02, -1.93008564e-03, ...,
        -1.26698607e-03, -1.74168198e-03, -2.69778091e-03],
       [-2.81373151e-03,  5.37250418e-03,  5.19963576e-03, ...,
         6.19022333e-04,  4.10330323e-03,  2.40162087e-03],
       ...,
       [-2.20824730e-03,  3.40977173e-03,  2.34557032e-03, ...,
         8.44890223e-04,  4.70369242e-03,  5.67542290e-03],
       [ 1.68164691e-03, -3.38554899e-04, -8.75254138e-05, ...,
        -1.67618567e-05, -3.41568213e-05, -2.09258946e-04],
       [ 1.79700458e-04, -3.51232340e-03, -5.58323155e-03, ...,
        -1.09157980e-03, -1.25215828e-03, -2.87728072e-03]])

In [130]:
top_n_interpret = 10
top_ig_gcn_bert_values = np.argpartition(ig_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_bert_values = np.argpartition(shap_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]

In [131]:
top_ig_gcn_bert_values = np.vectorize(map_to_idx)(top_ig_gcn_bert_values)
top_shap_gcn_bert_values = np.vectorize(map_to_idx)(top_shap_gcn_bert_values)

In [132]:
ig_gcn_bert_df = pd.DataFrame(top_ig_gcn_bert_values, index=test_idx)
shap_gcn_bert_df = pd.DataFrame(top_shap_gcn_bert_values, index=test_idx)

In [133]:
ig_gcn_bert_df = pd.melt(ig_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
shap_gcn_bert_df = pd.melt(shap_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")

In [134]:
labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.rel_id])
ig_gcn_bert_df["label"] = labels
ig_gcn_bert_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.rel_id])
shap_gcn_bert_df["label"] = labels
shap_gcn_bert_df["rel_label"] = rel_labels

In [135]:
ig_gcn_bert_source_df = ig_gcn_bert_df[["id", "label"]].drop_duplicates()
ig_gcn_bert_target_df = ig_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
ig_gcn_bert_node_df = pd.concat([ig_gcn_bert_source_df, ig_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

shap_gcn_bert_source_df = shap_gcn_bert_df[["id", "label"]].drop_duplicates()
shap_gcn_bert_target_df = shap_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
shap_gcn_bert_node_df = pd.concat([shap_gcn_bert_source_df, shap_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

In [137]:
ig_gcn_bert_G=nx.from_pandas_edgelist(ig_gcn_bert_df, "id", 'rel_id')#, create_using=nx.DiGraph)
shap_gcn_bert_G=nx.from_pandas_edgelist(shap_gcn_bert_df, "id", 'rel_id')#, create_using=nx.DiGraph)

In [138]:
ig_gcn_bert_id_df = ig_gcn_bert_df[["id", "label"]]
ig_gcn_bert_rel_df = ig_gcn_bert_df[["rel_id", "rel_label"]]
shap_gcn_bert_id_df = shap_gcn_bert_df[["id", "label"]]
shap_gcn_bert_rel_df = shap_gcn_bert_df[["rel_id", "rel_label"]]

In [139]:
new_columns = ["id", "label"]
ig_gcn_bert_id_df.columns = new_columns
ig_gcn_bert_rel_df.columns = new_columns
shap_gcn_bert_id_df.columns = new_columns
shap_gcn_bert_rel_df.columns = new_columns

In [140]:
ig_gcn_bert_id_rel_df = pd.concat([ig_gcn_bert_id_df, ig_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
ig_gcn_bert_id2label = dict(zip(ig_gcn_bert_id_rel_df.id, ig_gcn_bert_id_rel_df.label))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2label, "label")

shap_gcn_bert_id_rel_df = pd.concat([shap_gcn_bert_id_df, shap_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
shap_gcn_bert_id2label = dict(zip(shap_gcn_bert_id_rel_df.id, shap_gcn_bert_id_rel_df.label))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2label, "label")

In [141]:
ig_gcn_bert_id2text = {id: dataset.texts[id] for id in ig_gcn_bert_id_rel_df.id}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2text, "text")

shap_gcn_bert_id2text = {id: dataset.texts[id] for id in shap_gcn_bert_id_rel_df.id}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2text, "text")

In [142]:
ig_gcn_bert_id2drug = {node: ig_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in ig_gcn_bert_G.nodes()}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2drug, "drug")

shap_gcn_bert_id2drug = {node: shap_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in shap_gcn_bert_G.nodes()}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2drug, "drug")

In [143]:
(nx.is_connected(ig_gcn_bert_G), nx.number_connected_components(ig_gcn_bert_G),
 nx.is_connected(shap_gcn_bert_G), nx.number_connected_components(shap_gcn_bert_G))

(False, 19, False, 7)

In [149]:
ig_gcn_bert_components = nx.connected_components(ig_gcn_bert_G)
shap_gcn_bert_components = nx.connected_components(shap_gcn_bert_G)
[len(x) for x in ig_gcn_bert_components], [len(x) for x in shap_gcn_bert_components]

([1912,
  10,
  10,
  10,
  11,
  10,
  12,
  12,
  10,
  10,
  10,
  12,
  10,
  10,
  10,
  10,
  10,
  10,
  10],
 [2080, 20, 10, 10, 10, 10, 10])

In [151]:
ig_gcn_bert_degree_dict = dict(ig_gcn_bert_G.degree(ig_gcn_bert_G.nodes()))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_degree_dict, 'degree')
ig_gcn_bert_sorted_degree = sorted(ig_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_bert_degree_dict = dict(shap_gcn_bert_G.degree(shap_gcn_bert_G.nodes()))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_degree_dict, 'degree')
shap_gcn_bert_sorted_degree = sorted(shap_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_sorted_degree[:10], shap_gcn_bert_sorted_degree[:10]

([(1354, 35),
  (632, 33),
  (1372, 31),
  (123, 31),
  (1991, 26),
  (1355, 23),
  (436, 23),
  (373, 22),
  (1990, 22),
  (1837, 21)],
 [(1354, 35),
  (632, 34),
  (1372, 31),
  (123, 31),
  (1355, 26),
  (1991, 24),
  (1990, 22),
  (436, 22),
  (373, 19),
  (88, 19)])

In [152]:
ig_gcn_bert_betweenness_dict = nx.betweenness_centrality(ig_gcn_bert_G)
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_betweenness_dict, 'betweenness')
ig_gcn_bert_sorted_betweenness = sorted(ig_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_bert_betweenness_dict = nx.betweenness_centrality(shap_gcn_bert_G)
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_betweenness_dict, 'betweenness')
shap_gcn_bert_sorted_betweenness = sorted(shap_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_sorted_betweenness[:10], shap_gcn_bert_sorted_betweenness[:10]

([(109, 0.09839269169728862),
  (326, 0.06168925391759462),
  (474, 0.05382581533976217),
  (475, 0.05382581533976217),
  (159, 0.051263189945711884),
  (149, 0.04814998538681836),
  (382, 0.04757603095571717),
  (1444, 0.04538245114366873),
  (1991, 0.04409662323326127),
  (123, 0.04359103710084266)],
 [(1354, 0.07434279745073451),
  (334, 0.0500736538447221),
  (490, 0.047908685131070654),
  (333, 0.047293039993454696),
  (308, 0.047114152248241614),
  (326, 0.04618339834002428),
  (1991, 0.045667743035976725),
  (123, 0.041978155065546134),
  (844, 0.0365814193633946),
  (2447, 0.036211257869484696)])

In [153]:
ig_gcn_bert_communities = nx.community.greedy_modularity_communities(ig_gcn_bert_G)
ig_gcn_bert_modularity_dict = {}
for i, c in enumerate(ig_gcn_bert_communities):
    for name in c:
        ig_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_modularity_dict, 'community')

shap_gcn_bert_communities = nx.community.greedy_modularity_communities(shap_gcn_bert_G)
shap_gcn_bert_modularity_dict = {}
for i, c in enumerate(shap_gcn_bert_communities):
    for name in c:
        shap_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_modularity_dict, 'community')

len(ig_gcn_bert_communities), len(shap_gcn_bert_communities)

(59, 49)

In [154]:
([Counter([ig_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_bert_communities[:10]])

([0.1871345029239766,
  0.26666666666666666,
  0.8269230769230769,
  0.8333333333333334,
  0.6129032258064516,
  0.9090909090909091,
  0.6756756756756757,
  0.75,
  0.8333333333333334,
  0.7419354838709677],
 [0.20552147239263804,
  0.40268456375838924,
  0.32,
  0.43103448275862066,
  0.53,
  0.9381443298969072,
  0.9072164948453608,
  0.5405405405405406,
  0.8472222222222222,
  0.7083333333333334])

In [155]:
([Counter([ig_gcn_bert_G.nodes()[id]["drug"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["drug"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_bert_communities[:10]])

([0.10526315789473684,
  0.11666666666666667,
  0.125,
  0.10416666666666667,
  0.10752688172043011,
  0.09090909090909091,
  0.12162162162162163,
  0.09722222222222222,
  0.08333333333333333,
  0.11290322580645161],
 [0.0736196319018405,
  0.08053691275167785,
  0.112,
  0.1896551724137931,
  0.11,
  0.13402061855670103,
  0.1134020618556701,
  0.12162162162162163,
  0.09722222222222222,
  0.16666666666666666])

In [156]:
([Counter([ig_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in shap_gcn_bert_communities[:10]])

([('DM_Insulin und Tabletten', 64),
  ('Cholesterinsenker_KHK', 48),
  ('Blutdrucksenker_beides', 86),
  ('Blutdrucksenker_beides', 80),
  ('Blutdrucksenker_beides', 57),
  ('Blutdrucksenker_Blutdruck', 70),
  ('Blutdrucksenker_Blutdruck', 50),
  ('Blutdrucksenker_Blutdruck', 54),
  ('Blutdrucksenker_Blutdruck', 60),
  ('Blutdrucksenker_Blutdruck', 46)],
 [('Cholesterinsenker_beides', 67),
  ('Blutdrucksenker_beides', 60),
  ('Blutdrucksenker_Blutdruck', 40),
  ('Blutdrucksenker_Blutdruck', 50),
  ('Blutdrucksenker_beides', 53),
  ('Blutdrucksenker_beides', 91),
  ('Blutdrucksenker_Blutdruck', 88),
  ('Blutdrucksenker_beides', 40),
  ('Blutdrucksenker_Blutdruck', 61),
  ('Blutdrucksenker_beides', 51)])

In [183]:
train_count = 0
val_count = 0
test_count = 0

for node in ig_gcn_bert_G.nodes():
    if node not in test_idx: 
        continue
    for n in ig_gcn_bert_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s

(0.6767608029942157, 0.0, 0.3232391970057843)

In [184]:
train_count = 0
val_count = 0
test_count = 0

for node in shap_gcn_bert_G.nodes():
    if node not in test_idx: 
        continue
    for n in shap_gcn_bert_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s

(0.6710191622859081, 0.0, 0.3289808377140919)